# MEDALLIO ENTERPRISE
## AI & Analytics Operating System
### Notebook corporativo para Dirección, Data, Analytics, ML, Forecasting, Decision Intelligence y Gobierno

> **Objetivo:** convertir `medallio_dw` en un sistema corporativo de inteligencia que conecte datos, modelos, decisiones, control y creación de valor.

Este notebook eleva Medallio desde un laboratorio de Data Science hacia una **plataforma de gestión de IA empresarial**, con cinco preguntas rectoras:

1. **¿Qué está ocurriendo?** — observabilidad y KPIs.
2. **¿Por qué está ocurriendo?** — analytics, econometría y causalidad.
3. **¿Qué ocurrirá probablemente?** — ML y forecasting.
4. **¿Qué deberíamos hacer?** — Decision Intelligence y escenarios.
5. **¿Cómo sabemos que funcionó?** — experimentación, outcomes, ROI y gobierno.

---

## Arquitectura de valor

```text
FUENTES / CRM / MERCADO / FINANZAS / OPERACIÓN
                    │
                    ▼
          ┌──────────────────┐
          │ DATA FOUNDATION  │
          │ calidad · linaje │
          │ seguridad · SLA  │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │   ANALYTICS      │
          │ KPIs · cohortes  │
          │ mix · pricing    │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ ECONOMETRÍA / ML │
          │ explicación      │
          │ predicción       │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ DECISION INTEL.  │
          │ escenarios       │
          │ priorización     │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ VALUE REALIZATION│
          │ outcome · ROI    │
          │ aprendizaje      │
          └────────┬─────────┘
                   └──────────────► vuelve a aprender
```

---

## Estándar corporativo que persigue este notebook

- **Read-only por defecto.**
- **Evidence-first:** ninguna recomendación sin evidencia.
- **Time-aware:** splits y backtests temporales cuando corresponde.
- **Control de riesgo:** datos, modelos, seguridad y cambios.
- **Auditabilidad:** cada artefacto puede exportarse.
- **Valor medible:** toda iniciativa de IA debe vincularse con una decisión y un resultado económico/operativo.
- **Human-in-the-loop:** IA recomienda; la gobernanza define quién aprueba y quién responde.

# 0. Cómo ejecutar

Desde el repositorio:

```powershell
cd C:\Projectsd_replica_crm
.\.venv\Scripts\Activate.ps1
python -m pip install jupyterlab ipykernel pandas numpy matplotlib scikit-learn "psycopg[binary]" openpyxl pyarrow nbformat
python -m jupyter lab
```

Kernel recomendado: **Python - Medallio DW**.

In [ ]:
# Opcional: dependencias corporativas adicionales
# %pip install statsmodels scipy

# 1. Configuración corporativa y conexión segura

In [ ]:
from __future__ import annotations

import os, re, json, math, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RUN_TS = datetime.now(timezone.utc)
ARTIFACT_DIR = Path("artifacts") / "medallio_enterprise"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Run UTC:", RUN_TS.isoformat())
print("Artifacts:", ARTIFACT_DIR.resolve())

In [ ]:
def find_repo_root(start=None):
    p = (Path(start) if start else Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / ".git").exists() or (candidate / ".env.example").exists():
            return candidate
    return p

ROOT = find_repo_root()
ENV_PATH = ROOT / ".env"

def load_dotenv_minimal(path):
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        raw = raw.strip()
        if not raw or raw.startswith("#") or "=" not in raw:
            continue
        k, v = raw.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_dotenv_minimal(ENV_PATH)

DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "dbname": os.getenv("POSTGRES_DATABASE", "medallio_dw"),
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", ""),
    "connect_timeout": int(os.getenv("POSTGRES_CONNECT_TIMEOUT", "10")),
}

print("Repo:", ROOT)
print(f"DB: {DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
print("Password configurado:", bool(DB_CONFIG["password"]))

In [ ]:
import psycopg

FORBIDDEN_SQL = re.compile(
    r"\b(insert|update|delete|drop|alter|truncate|create|grant|revoke|copy|call|do)\b",
    flags=re.I,
)

def get_conn():
    return psycopg.connect(**DB_CONFIG)

def query_df(sql, params=None, read_only=True):
    statement = sql.strip().rstrip(";")
    if read_only and FORBIDDEN_SQL.search(statement):
        raise ValueError("Guardrail corporativo read-only: operación de escritura bloqueada.")
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(statement, params or ())
            cols = [d.name for d in cur.description] if cur.description else []
            rows = cur.fetchall() if cur.description else []
    return pd.DataFrame(rows, columns=cols)

def safe_relation(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Relación no permitida: {name}")
    return name

def relation_exists(name):
    schema, rel = safe_relation(name).split(".", 1)
    q = """
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables WHERE table_schema=%s AND table_name=%s
        UNION ALL
        SELECT 1 FROM information_schema.views WHERE table_schema=%s AND table_name=%s
    ) AS ok
    """
    return bool(query_df(q, (schema, rel, schema, rel)).iloc[0, 0])

def columns_of(rel):
    schema, table = safe_relation(rel).split(".", 1)
    return query_df("""
        SELECT ordinal_position, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema=%s AND table_name=%s
        ORDER BY ordinal_position
    """, (schema, table))

try:
    display(query_df("SELECT current_database() AS database, now() AS postgres_now"))
except Exception as e:
    print("No se pudo conectar todavía:", type(e).__name__, e)

# 2. Charter corporativo de IA

Una organización madura no comienza por “qué algoritmo usamos”, sino por **qué decisiones queremos mejorar**.

In [ ]:
AI_CHARTER = pd.DataFrame([
    ["Crecimiento", "Aumentar conversión y ventas", "Priorizar demanda y acciones comerciales", "Revenue incremental", "Gerencia Comercial"],
    ["Pricing", "Capturar valor sin frenar absorción", "Ajustar precios / descuentos", "Margen + velocidad", "Gerencia General / Comercial"],
    ["Stock", "Reducir permanencia y riesgo de inventario", "Gestionar unidades lentas", "Meses de cobertura", "Comercial / Operaciones"],
    ["Forecast", "Reducir incertidumbre", "Planificar metas y recursos", "WAPE / Bias / cobertura", "Gerencia General"],
    ["Leads", "Usar mejor la capacidad comercial", "Priorizar contactos", "Lift / conversión", "CRM / Comercial"],
    ["Riesgo", "Evitar decisiones defectuosas", "Bloquear datos/modelos no confiables", "Incidentes / SLA", "Data & AI Governance"],
    ["Aprendizaje", "Convertir decisiones en evidencia", "Medir outcomes", "ROI / uplift", "AI CoE"],
], columns=["dominio", "objetivo", "decision", "north_star", "accountable"])

display(AI_CHARTER)

# 3. Operating Model: quién decide, quién construye y quién controla

In [ ]:
RACI = pd.DataFrame([
    ["Definir KPI y decisión", "A", "R", "C", "C", "I", "I"],
    ["Contrato de datos", "C", "C", "A/R", "C", "C", "I"],
    ["Feature engineering", "I", "C", "R", "A/R", "C", "I"],
    ["Entrenamiento de modelo", "I", "C", "C", "A/R", "C", "I"],
    ["Validación independiente", "I", "I", "C", "C", "A/R", "I"],
    ["Aprobación a producción", "A", "C", "C", "R", "R", "I"],
    ["Monitoreo", "I", "I", "R", "R", "A", "I"],
    ["Respuesta a incidentes", "A", "C", "R", "R", "R", "I"],
    ["Medición de valor", "A", "R", "C", "R", "C", "I"],
], columns=["actividad", "Sponsor", "Product Owner", "Data Engineering", "Data Science", "AI Governance", "Usuario final"])

display(RACI)

# 4. Mapa de capacidades empresariales

Medallio debe poder demostrar capacidades, no sólo tablas.

In [ ]:
CAPABILITIES = pd.DataFrame([
    ["Data Foundation", "Ingesta / réplica", "¿Los datos llegaron?", 1],
    ["Data Foundation", "Calidad", "¿Son utilizables?", 1],
    ["Data Foundation", "Linaje", "¿De dónde viene cada métrica?", 2],
    ["Analytics", "Descriptivo", "¿Qué pasó?", 1],
    ["Analytics", "Diagnóstico", "¿Por qué pasó?", 2],
    ["Econometría", "Estacionalidad", "¿Es efecto calendario o proyecto?", 2],
    ["Econometría", "Elasticidad", "¿Cómo responde demanda a precio?", 3],
    ["ML", "Clasificación", "¿Quién tiene mayor probabilidad?", 2],
    ["ML", "Regresión", "¿Cuánto / cuándo?", 2],
    ["ML", "No supervisado", "¿Qué segmentos/regímenes existen?", 2],
    ["Forecast", "Backtesting", "¿Predice mejor que un baseline?", 2],
    ["Decision Intelligence", "Escenarios", "¿Qué pasa si...?", 2],
    ["Decision Intelligence", "Optimización", "¿Qué acción maximiza valor?", 4],
    ["MLOps", "Registry / versionado", "¿Qué modelo está vigente?", 3],
    ["MLOps", "Drift", "¿El presente se parece al entrenamiento?", 3],
    ["Governance", "Model Risk", "¿Qué nivel de control requiere?", 3],
    ["Value", "Experimentación", "¿La acción causó mejora?", 4],
    ["Value", "ROI", "¿El sistema crea valor económico?", 4],
], columns=["capa","capacidad","pregunta","nivel_objetivo"])

display(CAPABILITIES)

# 5. Inventario del Data Estate

In [ ]:
try:
    catalog = query_df("""
    SELECT
        n.nspname AS schema_name,
        c.relname AS relation_name,
        CASE c.relkind
            WHEN 'r' THEN 'table'
            WHEN 'p' THEN 'partitioned_table'
            WHEN 'v' THEN 'view'
            WHEN 'm' THEN 'materialized_view'
            ELSE c.relkind::text
        END AS relation_type,
        pg_total_relation_size(c.oid) AS bytes_total,
        pg_size_pretty(pg_total_relation_size(c.oid)) AS size_pretty,
        COALESCE(s.n_live_tup,0) AS estimated_rows
    FROM pg_class c
    JOIN pg_namespace n ON n.oid=c.relnamespace
    LEFT JOIN pg_stat_user_tables s ON s.relid=c.oid
    WHERE n.nspname IN (
        'analytics','features','model_control','decision_intelligence',
        'staging','raw_cygnus','raw_mercado','etl_control','experiments'
    )
    AND c.relkind IN ('r','p','v','m')
    ORDER BY pg_total_relation_size(c.oid) DESC
    """)
    display(catalog.head(80))
except Exception as e:
    catalog = pd.DataFrame()
    print("Inventario no disponible:", e)

In [ ]:
if len(catalog):
    schema_summary = (catalog.groupby("schema_name", as_index=False)
        .agg(objetos=("relation_name","count"),
             bytes_total=("bytes_total","sum"),
             estimated_rows=("estimated_rows","sum")))
    schema_summary["GB"] = schema_summary["bytes_total"] / 1024**3
    display(schema_summary.sort_values("GB", ascending=False))

    fig, ax = plt.subplots(figsize=(10,4))
    tmp = schema_summary.sort_values("GB", ascending=False)
    ax.bar(tmp["schema_name"], tmp["GB"])
    ax.set_ylabel("GB")
    ax.set_title("Huella del Data Estate")
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout(); plt.show()

# 6. Contrato empresarial de inteligencia

In [ ]:
CONTRACT = [
    ("Entendimiento", "analytics.fact_evento_comercial"),
    ("Entendimiento", "analytics.snapshot_unidad_diario"),
    ("Entendimiento", "analytics.historial_oferta_unidad"),
    ("Entendimiento", "analytics.historial_etapa_proyecto"),
    ("Econometría", "analytics.v_econometria_cobertura"),
    ("Econometría", "analytics.v_econometria_serie_precios"),
    ("Econometría", "features.v_estacionalidad_panel"),
    ("ML", "features.v_dataset_unidad_entrenamiento"),
    ("ML", "features.v_contexto_mercado_por_corte"),
    ("Forecast", "features.commercial_forecast_snapshots"),
    ("Forecast", "model_control.commercial_forecast_runs"),
    ("Forecast", "analytics.commercial_forecast_predictions"),
    ("Forecast", "analytics.commercial_forecast_backtest"),
    ("Forecast", "analytics.v_commercial_forecast_current"),
    ("Forecast", "analytics.v_commercial_forecast_performance"),
    ("Gobierno", "analytics.v_commercial_forecast_coverage"),
    ("Gobierno", "analytics.v_commercial_forecast_monitoring"),
    ("Decisión", "decision_intelligence.commercial_forecast_goals"),
    ("Decisión", "decision_intelligence.commercial_forecast_actions"),
    ("Resultados", "analytics.commercial_forecast_outcomes"),
]

rows=[]
for capa,obj in CONTRACT:
    try:
        exists=relation_exists(obj)
    except Exception:
        exists=False
    rows.append([capa,obj,exists])

contract_df=pd.DataFrame(rows,columns=["capa","objeto","existe"])
contract_df["estado"]=np.where(contract_df["existe"],"OK","PENDIENTE")
display(contract_df)

coverage=(contract_df.groupby("capa",as_index=False)["existe"].mean())
coverage["cobertura_pct"]=(coverage["existe"]*100).round(1)
display(coverage[["capa","cobertura_pct"]])

# 7. Data Quality & Observability

In [ ]:
def profile_relation(rel, limit=30000):
    if not relation_exists(rel):
        return {"objeto":rel,"existe":False}
    df=query_df(f"SELECT * FROM {safe_relation(rel)} LIMIT {int(limit)}")
    return {
        "objeto":rel,
        "existe":True,
        "filas_muestra":len(df),
        "columnas":df.shape[1],
        "nulos_pct_promedio":float(df.isna().mean().mean()*100) if len(df) else np.nan,
        "duplicados_pct":float(df.duplicated().mean()*100) if len(df) else np.nan,
    }

profiles=[]
for rel in contract_df.loc[contract_df["existe"],"objeto"]:
    try:
        profiles.append(profile_relation(rel))
    except Exception as e:
        profiles.append({"objeto":rel,"existe":True,"error":type(e).__name__})

profiles_df=pd.DataFrame(profiles)

def dq_status(r):
    n=r.get("nulos_pct_promedio",np.nan); d=r.get("duplicados_pct",np.nan)
    if pd.isna(n) and pd.isna(d): return "SIN_EVALUAR"
    if (pd.notna(n) and n>35) or (pd.notna(d) and d>10): return "ROJO"
    if (pd.notna(n) and n>15) or (pd.notna(d) and d>3): return "AMBAR"
    return "VERDE"

if len(profiles_df):
    profiles_df["semaforo"]=profiles_df.apply(dq_status,axis=1)
    display(profiles_df)

# 8. Seguridad, privacidad y datos sensibles

Esta sección **no muestra valores sensibles**. Sólo detecta nombres de columnas que podrían requerir clasificación y controles.

In [ ]:
try:
    dictionary = query_df("""
    SELECT table_schema, table_name, ordinal_position, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema IN ('analytics','features','model_control','decision_intelligence','staging','raw_cygnus')
    ORDER BY table_schema, table_name, ordinal_position
    """)
except Exception:
    dictionary = pd.DataFrame(columns=["table_schema","table_name","ordinal_position","column_name","data_type"])

PII_HINTS = {
    "identity": ["dni","documento","ruc","passport","pasaporte"],
    "contact": ["telefono","celular","email","correo","direccion"],
    "person": ["nombre","apellido","cliente"],
    "financial": ["ingreso","sueldo","salario","cuota","credito"],
}

def classify_column(name):
    x=name.lower()
    cats=[]
    for cat,terms in PII_HINTS.items():
        if any(t in x for t in terms): cats.append(cat)
    return ",".join(cats) if cats else ""

if len(dictionary):
    dictionary["sensitive_hint"] = dictionary["column_name"].map(classify_column)
    sensitive = dictionary[dictionary["sensitive_hint"]!=""].copy()
    display(sensitive.head(100))
    print("Columnas potencialmente sensibles detectadas:", len(sensitive))
else:
    sensitive=pd.DataFrame()

# 9. Risk Tiering de casos de IA

In [ ]:
MODEL_RISK = pd.DataFrame([
    ["Forecast comercial", "T2", "Medio", "Afecta planificación y metas", "backtest + monitoring mensual"],
    ["Lead scoring", "T2", "Medio", "Prioriza trabajo comercial", "fairness + lift + drift"],
    ["Pricing recommendation", "T3", "Alto", "Impacta ingresos y margen", "validación independiente + límites"],
    ["Anomaly detection", "T1", "Bajo", "Sólo genera alertas", "revisión humana"],
    ["Generación de insights", "T1", "Bajo", "Asistencia analítica", "evidencia / citations internas"],
    ["Optimización automática de precio", "T4", "Crítico", "Decisión automatizada de alto impacto", "aprobación ejecutiva + controles reforzados"],
], columns=["caso","tier","riesgo","razon","control_minimo"])

display(MODEL_RISK)

# 10. KPI Tree corporativo

In [ ]:
KPI_TREE = pd.DataFrame([
    ["Enterprise Value","Revenue","Monto vendido","S/","resultado"],
    ["Enterprise Value","Margen","Margen bruto","%","resultado"],
    ["Commercial","Demanda","Leads únicos","#","driver"],
    ["Commercial","Conversión","Lead → Separación","%","driver"],
    ["Commercial","Conversión","Separación → Minuta","%","driver"],
    ["Commercial","Velocidad","Ventas por mes","#","driver"],
    ["Inventory","Stock","Stock disponible","#","estado"],
    ["Inventory","Cobertura","Meses a stock cero","meses","riesgo"],
    ["Pricing","Precio","Precio/m²","S//m²","driver"],
    ["Pricing","Discount","Descuento medio","%","driver"],
    ["Forecast","Accuracy","WAPE","%","control"],
    ["Forecast","Bias","Sesgo","unidades","control"],
    ["ML","Discrimination","ROC-AUC / AP","score","control"],
    ["ML","Lift","Lift top decile","x","valor"],
    ["Governance","Reliability","SLO datasets/modelos","%","control"],
    ["Value","ROI","Beneficio incremental / costo IA","x","resultado"],
],columns=["nivel","tema","kpi","unidad","rol"])

display(KPI_TREE)

# 11. Explorador universal de datasets

In [ ]:
def inspect_relation(rel,n=8):
    if not relation_exists(rel):
        print("⏭ No existe:",rel); return None
    print("\n===",rel,"===")
    display(columns_of(rel))
    df=query_df(f"SELECT * FROM {safe_relation(rel)} LIMIT {int(n)}")
    display(df)
    return df

for rel in [
    "analytics.v_econometria_cobertura",
    "features.v_estacionalidad_panel",
    "features.v_dataset_unidad_entrenamiento",
    "analytics.v_commercial_forecast_current",
    "analytics.v_commercial_forecast_monitoring",
]:
    try: inspect_relation(rel,5)
    except Exception as e: print(rel,":",e)

# 12. Readiness de Machine Learning

In [ ]:
TARGET_HINTS=["target","label","vendido","venta","separacion","minuta","conversion","absorcion","demanda"]
DATE_HINTS=["fecha","date","periodo","mes","timestamp","captured_at","created_at"]

def infer_roles(df):
    rows=[]; n=max(len(df),1)
    for c in df.columns:
        s=df[c]; lc=c.lower(); nu=s.nunique(dropna=True)
        if pd.api.types.is_numeric_dtype(s): role="numeric"
        elif pd.api.types.is_datetime64_any_dtype(s) or any(x in lc for x in DATE_HINTS): role="date_candidate"
        elif nu/n>.90: role="id_candidate"
        else: role="categorical"
        score=max([1 if x==lc else .7 if x in lc else 0 for x in TARGET_HINTS]+[0])
        rows.append([c,str(s.dtype),role,nu,round(float(s.isna().mean()*100),2),score])
    return pd.DataFrame(rows,columns=["column","dtype","role","nunique","null_pct","target_hint"]).sort_values(["target_hint","role"],ascending=[False,True])

TRAIN_REL="features.v_dataset_unidad_entrenamiento"
train_raw=None
try:
    if relation_exists(TRAIN_REL):
        train_raw=query_df(f"SELECT * FROM {TRAIN_REL} LIMIT 200000")
        display(infer_roles(train_raw))
    else:
        print("⏭ Aún no existe",TRAIN_REL)
except Exception as e:
    print("Dataset ML no disponible:",e)

# 13. EDA corporativo

In [ ]:
def plot_numeric_distributions(df,max_cols=8):
    for c in df.select_dtypes(include=np.number).columns[:max_cols]:
        s=pd.to_numeric(df[c],errors="coerce").dropna()
        if len(s)<5: continue
        fig,ax=plt.subplots(figsize=(8,3.2)); ax.hist(s,bins=30)
        ax.set_title(f"Distribución — {c}"); ax.set_xlabel(c); ax.set_ylabel("Frecuencia")
        plt.tight_layout(); plt.show()

def plot_top_categories(df,max_cols=5,top_n=12):
    cats=[c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique(dropna=True)<=100][:max_cols]
    for c in cats:
        vc=df[c].astype("string").fillna("<NA>").value_counts().head(top_n).sort_values()
        fig,ax=plt.subplots(figsize=(8,4)); ax.barh(vc.index,vc.values)
        ax.set_title(f"Top categorías — {c}"); plt.tight_layout(); plt.show()

def plot_correlation_matrix(df,max_cols=15):
    nums=df.select_dtypes(include=np.number)
    if nums.shape[1]<2:return
    cols=nums.var(numeric_only=True).sort_values(ascending=False).head(max_cols).index
    corr=nums[cols].corr()
    fig,ax=plt.subplots(figsize=(10,8)); im=ax.imshow(corr.to_numpy(),aspect="auto",vmin=-1,vmax=1)
    ax.set_xticks(range(len(corr.columns)),corr.columns,rotation=90); ax.set_yticks(range(len(corr.index)),corr.index)
    ax.set_title("Correlaciones principales"); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

if train_raw is not None and len(train_raw):
    plot_numeric_distributions(train_raw)
    plot_top_categories(train_raw)
    plot_correlation_matrix(train_raw)

# 14. Comparabilidad interproyecto y estacionalidad

Estructura econométrica recomendada:

\[
Y_{p,t}=lpha_p+eta_1 EdadProyecto_{p,t}+\gamma_{mes(t)}+\delta X_{p,t}+arepsilon_{p,t}
\]

Esto separa:

- heterogeneidad estructural del proyecto,
- ciclo de vida comercial,
- estacionalidad calendario,
- composición de producto,
- precio, stock y contexto.

In [ ]:
SEASON_REL="features.v_estacionalidad_panel"
season_df=None
try:
    if relation_exists(SEASON_REL):
        season_df=query_df(f"SELECT * FROM {SEASON_REL} LIMIT 300000")
        display(season_df.head())
        display(infer_roles(season_df))
    else:
        print("⏭ Aún no existe",SEASON_REL)
except Exception as e:
    print("Panel no disponible:",e)

# 15. Econometría hands-on opcional

In [ ]:
# Este bloque intenta un OLS exploratorio si statsmodels está instalado y si defines las columnas.
# No reemplaza una especificación causal formal.

ECON_TARGET = None      # ej. "ventas_mes"
ECON_FEATURES = []      # ej. ["edad_proyecto_mes", "stock_inicio", "precio_m2"]

if season_df is not None and ECON_TARGET and ECON_FEATURES:
    try:
        import statsmodels.api as sm
        cols=[ECON_TARGET]+ECON_FEATURES
        econ=season_df[cols].copy().apply(pd.to_numeric,errors="coerce").dropna()
        X=sm.add_constant(econ[ECON_FEATURES])
        model=sm.OLS(econ[ECON_TARGET],X).fit(cov_type="HC3")
        print(model.summary())
    except Exception as e:
        print("OLS no ejecutado:",e)
else:
    print("Define ECON_TARGET y ECON_FEATURES para el laboratorio econométrico.")

# 16. ML supervisado con baseline y split temporal

In [ ]:
TARGET=None  # define explícitamente

if train_raw is not None:
    print("Candidatos de target:")
    display(infer_roles(train_raw).query("target_hint > 0").head(25))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance


def choose_time_column(df):
    cand=[]
    for c in df.columns:
        if any(x in c.lower() for x in DATE_HINTS):
            p=pd.to_datetime(df[c],errors="coerce")
            if p.notna().mean()>.60:cand.append((c,p.notna().mean()))
    return sorted(cand,key=lambda x:x[1],reverse=True)[0][0] if cand else None

def temporal_or_random_split(df,test_size=.2):
    tc=choose_time_column(df)
    if tc:
        t=pd.to_datetime(df[tc],errors="coerce")
        order=np.argsort(t.fillna(pd.Timestamp.min).to_numpy()); cut=int(len(df)*(1-test_size))
        return order[:cut],order[cut:],f"TEMPORAL:{tc}"
    idx=np.arange(len(df)); tr,te=train_test_split(idx,test_size=test_size,random_state=42)
    return tr,te,"ALEATORIO_FALLBACK"

model_bundle=None

In [ ]:
if train_raw is None or not len(train_raw):
    print("⏭ Sin dataset de entrenamiento.")
elif TARGET is None:
    print("🟡 Define TARGET.")
elif TARGET not in train_raw.columns:
    print("🔴 TARGET no existe.")
else:
    df=train_raw.copy(); y=df[TARGET]; X=df.drop(columns=[TARGET])

    # IDs casi únicos fuera del baseline.
    id_like=[c for c in X.columns if X[c].nunique(dropna=True)/max(len(X),1)>.98]
    X=X.drop(columns=id_like,errors="ignore")
    date_like=[c for c in X.columns if any(x in c.lower() for x in DATE_HINTS)]
    X=X.drop(columns=date_like,errors="ignore")

    mask=y.notna(); X=X.loc[mask].reset_index(drop=True); y=y.loc[mask].reset_index(drop=True)
    num_cols=X.select_dtypes(include=np.number).columns.tolist(); cat_cols=[c for c in X.columns if c not in num_cols]

    prep=ColumnTransformer([
        ("num",Pipeline([("imputer",SimpleImputer(strategy="median"))]),num_cols),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",min_frequency=10))]),cat_cols),
    ])

    uy=pd.Series(y).dropna().nunique()
    is_classification=uy<=12 and (pd.api.types.is_bool_dtype(y) or uy<=3 or set(pd.Series(y).dropna().unique()).issubset({0,1,True,False}))
    tr,te,split_kind=temporal_or_random_split(X)

    estimator=(HistGradientBoostingClassifier(learning_rate=.06,max_iter=250,max_leaf_nodes=31,random_state=42)
               if is_classification else
               HistGradientBoostingRegressor(learning_rate=.06,max_iter=250,max_leaf_nodes=31,random_state=42))
    pipe=Pipeline([("prep",prep),("model",estimator)]); pipe.fit(X.iloc[tr],y.iloc[tr]); pred=pipe.predict(X.iloc[te])

    if is_classification:
        prob=pipe.predict_proba(X.iloc[te])[:,1]
        metrics={"split":split_kind,"n_train":len(tr),"n_test":len(te),
                 "roc_auc":roc_auc_score(y.iloc[te],prob) if y.iloc[te].nunique()>1 else np.nan,
                 "average_precision":average_precision_score(y.iloc[te],prob) if y.iloc[te].nunique()>1 else np.nan,
                 "brier":brier_score_loss(y.iloc[te],prob)}
    else:
        a=pd.to_numeric(y.iloc[te],errors="coerce"); p=pd.to_numeric(pd.Series(pred),errors="coerce")
        metrics={"split":split_kind,"n_train":len(tr),"n_test":len(te),
                 "mae":mean_absolute_error(a,p),"rmse":mean_squared_error(a,p)**.5,
                 "r2":r2_score(a,p),"wape":np.abs(a-p).sum()/max(np.abs(a).sum(),1e-9)}

    model_bundle={"pipeline":pipe,"X":X,"y":y,"test_idx":te,"is_classification":is_classification,"metrics":metrics}
    display(pd.DataFrame([metrics]))

# 17. Explicabilidad / Feature Importance

In [ ]:
if model_bundle:
    pipe=model_bundle["pipeline"]; X=model_bundle["X"]; y=model_bundle["y"]; te=model_bundle["test_idx"]
    sample=te[-min(len(te),5000):]
    scoring="roc_auc" if model_bundle["is_classification"] and pd.Series(y.iloc[sample]).nunique()>1 else None
    try:
        perm=permutation_importance(pipe,X.iloc[sample],y.iloc[sample],n_repeats=4,random_state=42,scoring=scoring)
        imp=pd.DataFrame({"feature":X.columns,"importance_mean":perm.importances_mean,"importance_std":perm.importances_std}).sort_values("importance_mean",ascending=False)
        display(imp.head(30))
        top=imp.head(20).sort_values("importance_mean")
        fig,ax=plt.subplots(figsize=(8,6)); ax.barh(top["feature"],top["importance_mean"]); ax.set_title("Importancia por permutación")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print("Importancia no disponible:",e)
else:
    print("⏭ Entrena un modelo primero.")

# 18. Segmentación y regímenes

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler

cluster_result=None
if train_raw is not None and len(train_raw)>=100:
    numeric=train_raw.select_dtypes(include=np.number).copy().dropna(axis=1,thresh=int(len(train_raw)*.6))
    keep=[c for c in numeric.columns if numeric[c].nunique(dropna=True)>3 and numeric[c].nunique(dropna=True)/len(numeric)<.95][:20]
    if len(keep)>=2:
        Z=numeric[keep].replace([np.inf,-np.inf],np.nan); Z=Z.fillna(Z.median()); Zs=RobustScaler().fit_transform(Z)
        K=4 if len(Z)>=200 else 3
        labels=KMeans(n_clusters=K,random_state=42,n_init=10).fit_predict(Zs)
        cluster_result=Z.copy(); cluster_result["cluster"]=labels
        display(cluster_result.groupby("cluster")[keep].median().T)
    else: print("⏭ Variables insuficientes.")
else: print("⏭ Dataset insuficiente.")

# 19. Detección de anomalías

In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_result=None
if train_raw is not None and len(train_raw)>=100:
    numeric=train_raw.select_dtypes(include=np.number).copy().dropna(axis=1,thresh=int(len(train_raw)*.7))
    keep=[c for c in numeric.columns if numeric[c].nunique(dropna=True)>5 and numeric[c].nunique(dropna=True)/len(numeric)<.98][:25]
    if len(keep)>=2:
        Z=numeric[keep].replace([np.inf,-np.inf],np.nan).fillna(numeric[keep].median())
        iso=IsolationForest(n_estimators=300,contamination=.02,random_state=42,n_jobs=-1).fit(Z)
        anomaly_result=train_raw.loc[Z.index].copy(); anomaly_result["anomaly_score"]=-iso.score_samples(Z); anomaly_result["es_anomalia"]=iso.predict(Z)==-1
        display(anomaly_result.sort_values("anomaly_score",ascending=False).head(30))
        print("Anomalías:",int(anomaly_result["es_anomalia"].sum()))

# 20. Forecasting corporativo: desempeño, bias e incertidumbre

In [ ]:
FORECAST_CURRENT="analytics.v_commercial_forecast_current"
FORECAST_PERF="analytics.v_commercial_forecast_performance"
FORECAST_MON="analytics.v_commercial_forecast_monitoring"

fc=perf=fm=None
for name,var in [(FORECAST_CURRENT,"fc"),(FORECAST_PERF,"perf"),(FORECAST_MON,"fm")]:
    try:
        if relation_exists(name):
            data=query_df(f"SELECT * FROM {name}")
            globals()[var]=data
            print(name, len(data)); display(data.head(50))
        else: print("⏭",name)
    except Exception as e: print(name,e)

In [ ]:
if perf is not None and {"actual","prediction"}.issubset(perf.columns):
    ev=perf.dropna(subset=["actual","prediction"]).copy()
    if len(ev):
        a=pd.to_numeric(ev["actual"],errors="coerce"); p=pd.to_numeric(ev["prediction"],errors="coerce")
        score=pd.DataFrame([{"n_maduros":len(ev),"MAE":np.mean(np.abs(p-a)),"RMSE":np.sqrt(np.mean((p-a)**2)),"BIAS":np.mean(p-a),"WAPE":np.abs(p-a).sum()/max(np.abs(a).sum(),1e-9)}])
        display(score)
        fig,ax=plt.subplots(figsize=(6,6)); ax.scatter(a,p,alpha=.5); lo=min(a.min(),p.min()); hi=max(a.max(),p.max()); ax.plot([lo,hi],[lo,hi],linestyle="--")
        ax.set_xlabel("Real"); ax.set_ylabel("Pronóstico"); ax.set_title("Forecast vs resultado"); plt.tight_layout(); plt.show()

# 21. Challenger Framework

In [ ]:
def forecast_metrics(actual,pred):
    a=pd.Series(actual,dtype="float64"); p=pd.Series(pred,dtype="float64"); m=a.notna()&p.notna(); a,p=a[m],p[m]
    if not len(a): return {}
    return {"MAE":float(np.mean(np.abs(p-a))),"RMSE":float(np.sqrt(np.mean((p-a)**2))),"BIAS":float(np.mean(p-a)),"WAPE":float(np.abs(p-a).sum()/max(np.abs(a).sum(),1e-9))}

CHALLENGERS = pd.DataFrame([
    ["Naive t-1","Último periodo","Baseline obligatorio"],
    ["Moving Avg 3","Promedio 3 meses","Suaviza ruido"],
    ["Seasonal Naive","Mismo mes año anterior","Captura calendario"],
    ["Lifecycle Peer","Mismo mes de vida de proyectos comparables","Captura madurez"],
    ["ML Forecast","Modelo actual","Debe superar challengers"],
],columns=["modelo","definicion","rol"])

display(CHALLENGERS)

# 22. Drift y estabilidad

In [ ]:
def psi_numeric(expected,actual,bins=10):
    e=pd.to_numeric(expected,errors="coerce").dropna(); a=pd.to_numeric(actual,errors="coerce").dropna()
    if len(e)<30 or len(a)<30:return np.nan
    edges=np.unique(np.quantile(e,np.linspace(0,1,bins+1)))
    if len(edges)<3:return np.nan
    edges[0],edges[-1]=-np.inf,np.inf
    ep=pd.cut(e,bins=edges,include_lowest=True).value_counts(normalize=True,sort=False).clip(lower=1e-6)
    ap=pd.cut(a,bins=edges,include_lowest=True).value_counts(normalize=True,sort=False).clip(lower=1e-6)
    return float(((ap-ep)*np.log(ap/ep)).sum())

def drift_report(df,time_col=None):
    w=df.copy()
    if time_col:
        t=pd.to_datetime(w[time_col],errors="coerce"); w=w.iloc[np.argsort(t.fillna(pd.Timestamp.min).to_numpy())].reset_index(drop=True)
    cut=int(len(w)*.7); old,new=w.iloc[:cut],w.iloc[cut:]
    out=pd.DataFrame([{"feature":c,"PSI":psi_numeric(old[c],new[c])} for c in w.select_dtypes(include=np.number).columns]).sort_values("PSI",ascending=False)
    out["semaforo"]=pd.cut(out["PSI"],[-np.inf,.10,.25,np.inf],labels=["ESTABLE","REVISAR","CAMBIO_FUERTE"])
    return out

drift=None
if train_raw is not None and len(train_raw)>=200:
    drift=drift_report(train_raw,choose_time_column(train_raw))
    display(drift.head(30))

# 23. Model Registry conceptual

In [ ]:
MODEL_REGISTRY_TEMPLATE = pd.DataFrame([
    ["lead_scoring","1.0.0","candidate","T2","Data Science","pendiente","mensual"],
    ["commercial_forecast","1.0.0","production","T2","Data Science","aprobado","mensual"],
    ["pricing_elasticity","0.1.0","research","T3","Econometría","pendiente","trimestral"],
], columns=["model_name","version","stage","risk_tier","owner","validation_status","review_frequency"])

display(MODEL_REGISTRY_TEMPLATE)

# 24. Model Card corporativa

In [ ]:
MODEL_CARD = {
    "model_name": None,
    "business_decision": None,
    "owner": None,
    "risk_tier": None,
    "training_window": None,
    "target": TARGET,
    "features": None,
    "baseline": None,
    "metrics": model_bundle["metrics"] if model_bundle else None,
    "known_limitations": [],
    "prohibited_uses": [],
    "human_approval_required": True,
    "monitoring": ["performance","drift","data_quality","coverage"],
    "rollback_plan": "Volver al baseline/regla previa",
}
print(json.dumps(MODEL_CARD,indent=2,ensure_ascii=False,default=str))

# 25. Escenarios y Decision Intelligence

In [ ]:
def scenario_grid(base_units,base_price,demand_changes=(-.20,-.10,-.05,0,.05,.10,.20),price_changes=(-.05,-.03,0,.03,.05)):
    rows=[]
    for dq in demand_changes:
        for dp in price_changes:
            units=max(base_units*(1+dq),0); price=max(base_price*(1+dp),0)
            rows.append([dq,dp,units,price,units*price])
    return pd.DataFrame(rows,columns=["delta_demanda","delta_precio","unidades","precio","revenue"])

BASE_UNITS=10; BASE_PRICE=500000
scenarios=scenario_grid(BASE_UNITS,BASE_PRICE)
pivot=scenarios.pivot(index="delta_demanda",columns="delta_precio",values="revenue")
display(pivot.style.format("{:,.0f}"))

fig,ax=plt.subplots(figsize=(8,5)); im=ax.imshow(pivot.values,aspect="auto")
ax.set_xticks(range(len(pivot.columns)),[f"{x:+.0%}" for x in pivot.columns]); ax.set_yticks(range(len(pivot.index)),[f"{x:+.0%}" for x in pivot.index])
ax.set_xlabel("Cambio precio"); ax.set_ylabel("Cambio demanda"); ax.set_title("Mapa de sensibilidad de revenue"); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

# 26. Causalidad y experimentación

In [ ]:
EXPERIMENT_TEMPLATE = pd.DataFrame([
    ["Hipótesis","Ej.: priorizar leads top score aumenta conversión"],
    ["Unidad experimental","Lead / asesor / proyecto"],
    ["Tratamiento","Acción comercial específica"],
    ["Control","Proceso habitual"],
    ["Primary metric","Conversión / revenue incremental"],
    ["Guardrails","Caídas, satisfacción, descuento, carga asesor"],
    ["Asignación","Aleatoria y persistente"],
    ["Duración","Definida por potencia / maduración"],
    ["Criterio de éxito","Uplift > umbral y sin romper guardrails"],
    ["Decision rule","Promover / iterar / retirar"],
],columns=["componente","definicion"])

display(EXPERIMENT_TEMPLATE)

# 27. Cola de decisiones ejecutivas

In [ ]:
decision_queue=pd.DataFrame(columns=["dominio","entidad","senal","impacto_estimado","confianza","urgencia","accion_recomendada","owner","evidencia"])

decision_queue.loc[len(decision_queue)]=[
    "Forecast","DEMO","Ritmo esperado debajo de meta",250000,.75,.80,
    "Revisar meta, pricing y presión comercial","Gerencia Comercial","forecast vs meta"
]

decision_queue["priority_score"]=(pd.to_numeric(decision_queue["impacto_estimado"],errors="coerce").fillna(0)*
                                  pd.to_numeric(decision_queue["confianza"],errors="coerce").fillna(0)*
                                  pd.to_numeric(decision_queue["urgencia"],errors="coerce").fillna(0))
display(decision_queue.sort_values("priority_score",ascending=False))

# 28. Value Realization: del modelo al ROI

In [ ]:
VALUE_CASES = pd.DataFrame([
    ["Lead scoring","Conversión incremental","S/ incremental - costo operativo","uplift vs control"],
    ["Forecast","Menor error de planificación","Costo evitado / menor inventario","vs baseline"],
    ["Pricing","Mayor ingreso/margen","Revenue/margen incremental","test / causalidad"],
    ["Stock alerts","Menor aging","capital liberado / descuentos evitados","antes-después controlado"],
    ["Automation","Horas ahorradas","horas x costo fully-loaded","time study"],
],columns=["caso","valor","formula","evidencia"])

display(VALUE_CASES)

In [ ]:
def value_score(benefit, cost, adoption=1.0, confidence=1.0):
    net = benefit*adoption*confidence - cost
    roi = net/cost if cost else np.nan
    return {"benefit_adjusted":benefit*adoption*confidence,"cost":cost,"net_value":net,"roi":roi}

example_value=value_score(benefit=500000,cost=120000,adoption=.70,confidence=.75)
display(pd.DataFrame([example_value]))

# 29. SLO / SLA del sistema de IA

In [ ]:
SLOS = pd.DataFrame([
    ["Replica CRM","Frescura","<= 90 min","cada hora","Data Engineering"],
    ["Dataset ML","Disponibilidad",">= 99%","diario","Data Engineering"],
    ["Forecast","Cobertura proyectos",">= 95%","mensual","Data Science"],
    ["Forecast","WAPE","<= umbral por horizonte","mensual","Data Science"],
    ["Lead scoring","Freshness","<= 24h","diario","Data Science"],
    ["Decision queue","Acciones vencidas","0 críticas","diario","Product Owner"],
    ["Model monitoring","Drift reviews","100% señales rojas","mensual","AI Governance"],
],columns=["producto","SLO","target","frecuencia","owner"])

display(SLOS)

# 30. Gestión de incidentes de IA

In [ ]:
INCIDENT_MATRIX = pd.DataFrame([
    ["SEV-1","Decisión materialmente incorrecta / datos corruptos en producción","Inmediato","desactivar modelo, volver a baseline, informar sponsor"],
    ["SEV-2","Degradación significativa / cobertura rota","< 4h","contener, evaluar impacto, corregir"],
    ["SEV-3","Drift o calidad amarilla","< 1 día","investigar y programar fix"],
    ["SEV-4","Mejora / deuda técnica","backlog","priorizar por valor"],
],columns=["severidad","definicion","respuesta","accion"])

display(INCIDENT_MATRIX)

# 31. Release Gates: cuándo un modelo puede avanzar

In [ ]:
RELEASE_GATES = pd.DataFrame([
    ["G0 Research","Problema y KPI definidos","Product Owner"],
    ["G1 Data Ready","Calidad, linaje, leakage revisados","Data Engineering"],
    ["G2 Model Ready","Baseline superado y backtest válido","Data Science"],
    ["G3 Risk Ready","Model card, tier, controles y limitaciones","AI Governance"],
    ["G4 Pilot","Piloto controlado + human-in-the-loop","Negocio"],
    ["G5 Production","SLO, monitoring, rollback, owner","Sponsor"],
    ["G6 Scale","Valor demostrado y adopción suficiente","Steering Committee"],
],columns=["gate","criterio","aprobador"])

display(RELEASE_GATES)

# 32. Portfolio de casos de IA

In [ ]:
AI_PORTFOLIO = pd.DataFrame([
    ["Lead scoring",8,7,6,"Build/Pilot"],
    ["Forecast comercial",9,8,7,"Scale"],
    ["Pricing elasticity",9,5,8,"Research"],
    ["Stock risk",8,8,5,"Build"],
    ["Anomaly detection",6,9,3,"Quick win"],
    ["Next best action",9,4,8,"Future"],
],columns=["use_case","business_value","data_readiness","risk_complexity","stage"])

AI_PORTFOLIO["priority"] = AI_PORTFOLIO["business_value"]*AI_PORTFOLIO["data_readiness"]/(AI_PORTFOLIO["risk_complexity"]+1)
display(AI_PORTFOLIO.sort_values("priority",ascending=False))

fig,ax=plt.subplots(figsize=(7,5)); ax.scatter(AI_PORTFOLIO["data_readiness"],AI_PORTFOLIO["business_value"],s=(AI_PORTFOLIO["risk_complexity"]+1)*60)
for _,r in AI_PORTFOLIO.iterrows(): ax.text(r["data_readiness"]+.05,r["business_value"]+.05,r["use_case"],fontsize=8)
ax.set_xlabel("Data readiness"); ax.set_ylabel("Business value"); ax.set_title("Portfolio IA: valor vs readiness (tamaño = complejidad/riesgo)")
plt.tight_layout(); plt.show()

# 33. AI FinOps / costo de servir

In [ ]:
FINOPS_TEMPLATE = pd.DataFrame([
    ["Compute ETL",0,"mensual"],
    ["Storage",0,"mensual"],
    ["Training",0,"por corrida"],
    ["Inference",0,"por 1k predicciones"],
    ["Monitoring",0,"mensual"],
    ["Licencias / SaaS",0,"mensual"],
    ["Horas equipo",0,"mensual"],
],columns=["componente","costo_S","unidad"])

display(FINOPS_TEMPLATE)

# 34. Madurez empresarial de IA

In [ ]:
# Score observable + score objetivo
maturity_rows=[]
for dim,filter_capa in [
    ("Datos",["Entendimiento"]),("Econometría",["Econometría"]),("ML",["ML"]),
    ("Forecast",["Forecast"]),("Gobierno",["Gobierno"]),("Decisión",["Decisión"]),("Resultados",["Resultados"])
]:
    s=contract_df[contract_df["capa"].isin(filter_capa)]["existe"]
    maturity_rows.append([dim,float(s.mean()) if len(s) else 0])

maturity=pd.DataFrame(maturity_rows,columns=["dimension","score_0_1"])
maturity["score_pct"]=(maturity["score_0_1"]*100).round(0)
maturity["target_pct"]=100

display(maturity)
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(maturity["dimension"],maturity["score_pct"]); ax.set_ylim(0,105); ax.set_ylabel("% madurez observable"); ax.set_title("Madurez empresarial de Medallio")
ax.tick_params(axis="x",rotation=30); plt.tight_layout(); plt.show()

# 35. Scorecard de gobierno corporativo

In [ ]:
GOV_SCORECARD = pd.DataFrame([
    ["Data ownership","¿Cada dataset crítico tiene owner?",False],
    ["Data contracts","¿Hay contratos/versionado?",False],
    ["Lineage","¿Métrica → fuente trazable?",False],
    ["Model registry","¿Modelo/version/stage registrados?",False],
    ["Model cards","¿Limitaciones y usos documentados?",False],
    ["Independent validation","¿T3/T4 tiene revisión independiente?",False],
    ["Monitoring","¿Performance/drift/coverage activos?",bool(relation_exists(FORECAST_MON)) if 'relation_exists' in globals() else False],
    ["Rollback","¿Existe baseline y plan de reversión?",True],
    ["Value tracking","¿Outcome y ROI medidos?",bool(relation_exists("analytics.commercial_forecast_outcomes")) if 'relation_exists' in globals() else False],
    ["Incident management","¿Hay severidad y respuesta?",True],
],columns=["control","pregunta","cumple"])

display(GOV_SCORECARD)
print("Governance score:",round(GOV_SCORECARD["cumple"].mean()*100,1),"%")

# 36. Executive AI Brief

In [ ]:
def build_executive_brief():
    present=int(contract_df["existe"].sum()); total=len(contract_df)
    missing=contract_df.loc[~contract_df["existe"],"objeto"].tolist()
    return {
        "timestamp_utc":RUN_TS.isoformat(),
        "enterprise_coverage_pct":round(100*present/max(total,1),1),
        "governance_score_pct":round(GOV_SCORECARD["cumple"].mean()*100,1),
        "model_trained_in_notebook":bool(model_bundle is not None),
        "model_metrics":model_bundle["metrics"] if model_bundle else None,
        "forecast_available":bool(fc is not None),
        "forecast_performance_available":bool(perf is not None),
        "missing_contract_objects":missing,
        "top_decisions":decision_queue.sort_values("priority_score",ascending=False).head(10).to_dict("records"),
        "top_ai_portfolio":AI_PORTFOLIO.sort_values("priority",ascending=False).head(10).to_dict("records"),
    }

brief=build_executive_brief()
print(json.dumps(brief,indent=2,ensure_ascii=False,default=str))

# 37. Steering Committee Pack

In [ ]:
STEERCO = pd.DataFrame([
    ["1. Business","KPIs, gap vs meta, riesgos y oportunidades"],
    ["2. Data","frescura, calidad, cobertura, incidentes"],
    ["3. Models","desempeño, drift, challengers, cambios"],
    ["4. Decisions","acciones priorizadas, owners, vencimientos"],
    ["5. Value","ROI, uplift, ahorro, adopción"],
    ["6. Risk","model risk, seguridad, excepciones"],
    ["7. Roadmap","qué escala, qué se pausa, qué entra"],
],columns=["seccion","contenido"])

display(STEERCO)

# 38. Roadmap 30-60-90 y 12 meses

In [ ]:
ROADMAP = pd.DataFrame([
    ["0-30 días","Foundation","Contratos de datos, owners, calidad, diccionario, SLA"],
    ["0-30 días","Governance","RACI, risk tiers, release gates, model card template"],
    ["31-60 días","ML","Baselines reproducibles, registry, feature definitions"],
    ["31-60 días","Forecast","Backtest por horizonte + challenger obligatorio"],
    ["61-90 días","Decision Intelligence","Cola de decisiones + outcomes"],
    ["61-90 días","Experimentación","Piloto lead scoring / pricing con control"],
    ["3-6 meses","MLOps","Automatizar scoring, monitoring, retraining gates"],
    ["3-6 meses","Econometría","Estacionalidad + elasticidad + panel interproyecto"],
    ["6-12 meses","Optimization","Next best action / optimización comercial"],
    ["6-12 meses","Scale","AI portfolio + ROI + governance recurrente"],
],columns=["horizonte","frente","resultado"])

display(ROADMAP)

# 39. Export Pack corporativo

In [ ]:
exports = {
    "ai_charter.csv":AI_CHARTER,
    "raci.csv":RACI,
    "capabilities.csv":CAPABILITIES,
    "contract_status.csv":contract_df,
    "data_quality.csv":profiles_df,
    "pii_inventory.csv":sensitive,
    "model_risk.csv":MODEL_RISK,
    "kpi_tree.csv":KPI_TREE,
    "model_registry_template.csv":MODEL_REGISTRY_TEMPLATE,
    "decision_queue.csv":decision_queue,
    "value_cases.csv":VALUE_CASES,
    "slos.csv":SLOS,
    "incident_matrix.csv":INCIDENT_MATRIX,
    "release_gates.csv":RELEASE_GATES,
    "ai_portfolio.csv":AI_PORTFOLIO,
    "maturity.csv":maturity,
    "governance_scorecard.csv":GOV_SCORECARD,
    "roadmap.csv":ROADMAP,
}

if len(dictionary): exports["data_dictionary.csv"] = dictionary
if drift is not None: exports["drift_psi.csv"] = drift
if fc is not None: exports["forecast_current.csv"] = fc
if perf is not None: exports["forecast_performance.csv"] = perf

for name,df in exports.items():
    try: df.to_csv(ARTIFACT_DIR/name,index=False)
    except Exception as e: print("No exportado",name,e)

(ARTIFACT_DIR/"executive_ai_brief.json").write_text(json.dumps(brief,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
(ARTIFACT_DIR/"model_card.json").write_text(json.dumps(MODEL_CARD,indent=2,ensure_ascii=False,default=str),encoding="utf-8")

print("Artifacts generados:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(" -",p)

# 40. Definición de “nivel corporativo” para Medallio

Medallio alcanza un nivel corporativo cuando puede responder, de forma reproducible y auditable:

### Negocio
- ¿Qué KPI se quiere mover?
- ¿Quién es accountable?
- ¿Qué decisión concreta cambia?
- ¿Cuál es el impacto financiero esperado?

### Datos
- ¿De dónde sale el dato?
- ¿Está fresco y completo?
- ¿Quién es dueño del dataset?
- ¿Qué información sensible contiene?

### Ciencia
- ¿Cuál es el baseline?
- ¿El modelo supera el baseline fuera de muestra?
- ¿Hay leakage?
- ¿Qué variables explican el resultado?
- ¿Qué incertidumbre tiene?

### Riesgo
- ¿Qué pasa si el modelo falla?
- ¿Quién puede aprobarlo?
- ¿Existe rollback?
- ¿Cómo se detecta drift?

### Operación
- ¿Con qué frecuencia corre?
- ¿Qué SLO debe cumplir?
- ¿Quién recibe la recomendación?
- ¿Qué acción queda registrada?

### Valor
- ¿Se tomó la acción?
- ¿Qué outcome ocurrió?
- ¿Hubo uplift?
- ¿Cuál fue el ROI?

---

# North Star final

> **Medallio no debe ser un repositorio de datos ni una colección de modelos. Debe ser el sistema operativo que transforma evidencia en decisiones, registra esas decisiones y aprende de sus resultados.**